# Day 5 · Notebook 4 of 5 – ML extension with MLflow (next-day failure risk)

**Goal:** a small pilot that shows the full MLflow flow on Databricks:

`feature table → train 2 models → log to MLflow → pick the best → register in Unity Catalog → set alias "champion" → score machines → dashboard view`

| Step | Output |
|---|---|
| 1 | Feature table `gold.ml_machine_daily_features` |
| 2 | Quick look at the training data |
| 3 | Train logistic regression and random forest, log both runs to MLflow |
| 4 | Log the best model, register it as `gold.failure_risk_model`, set alias `champion` |
| 5 | Score all machines into `gold.machine_failure_risk` and create `gold.vw_machine_failure_risk` |
| 6 | Limitations |

**Run after** `7_1_dashboard_views` (it uses `gold.vw_sensor_drift` and `gold.vw_machine_reliability`).
**Cluster:** Databricks Runtime ML (has scikit-learn and MLflow).

## Setup

**What this code does (in simple words)**
* Loads the Python tools we need: pandas/numpy for tables and numbers, matplotlib for charts, MLflow for tracking and saving models, and scikit-learn for building the models.
* Sets the names we use everywhere: the gold and silver schemas, the feature table, the risk table, the model name and the label column (`label_failure_next_day`).
* Tells MLflow to save models in **Unity Catalog**.
* Creates (or reuses) an MLflow **experiment** in your user folder, which is the place where every training run is recorded.

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from mlflow import MlflowClient
from mlflow.models import infer_signature
from pyspark.sql import functions as F

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay)

GOLD, SILVER = "capstone_5iot.gold", "capstone_5iot.silver"
FEATURE_TABLE = f"{GOLD}.ml_machine_daily_features"
RISK_TABLE    = f"{GOLD}.machine_failure_risk"
MODEL_NAME    = f"{GOLD}.failure_risk_model"
LABEL         = "label_failure_next_day"

mlflow.set_registry_uri("databricks-uc")              # register models in Unity Catalog
user = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/capstone05_failure_risk")

## Step 1 – Feature table (one row per machine per day)
* **Features** describe the machine at the end of the day: readings, abnormal count and rate, failures and downtime today, daily sensor averages and maxima, drift flags, days since the last completed work order, criticality and machine type.
* **Label** `label_failure_next_day` = 1 if the machine fails the next day, else 0.
* The failure log only covers 2026-07-01 → 2026-07-02. After that, "no failure" means *no data*, not "no failure". So the label is only filled where the next day is inside the failure log, otherwise it stays **NULL**. NULL rows are not used for training, only for scoring.

* Builds a list of **every machine for every day** (machine x date).
* For each machine-day it adds facts about that day: number of sensor readings, abnormal readings, failures and downtime, average and maximum temperature / vibration / pressure, whether a sensor was drifting, and how many days have passed since the last finished maintenance job.
* It adds the **answer column** (label): 1 if the machine failed the **next day**, 0 if it did not. If the next day is outside the dates covered by the failure log, the label is left empty (NULL), because we simply do not know.
* Saves all of this as the table `gold.ml_machine_daily_features`.
* Finally shows a quick count: total rows, number of machines, how many rows have a label and how many of those are failures.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {FEATURE_TABLE}
COMMENT 'Day 5 ML features per machine per day; label = failure next day (NULL outside the failure log window)'
AS
WITH grid AS (            -- every machine x every day
  SELECT m.machine_id, m.criticality, m.machine_type, d.date AS activity_date
  FROM {GOLD}.dim_machine m CROSS JOIN {GOLD}.dim_date d
),
failure_window AS (       -- first and last day covered by the failure log
  SELECT min(to_date(failure_ts)) AS first_day, max(to_date(failure_ts)) AS last_day
  FROM {GOLD}.fact_maintenance_episodes
),
failure_days AS (
  SELECT DISTINCT machine_id, to_date(failure_ts) AS fail_date FROM {GOLD}.fact_maintenance_episodes
),
last_maintenance AS (     -- latest completed work order up to that day (no future information)
  SELECT g.machine_id, g.activity_date, max(w.completed_ts_clean) AS last_maint_ts
  FROM grid g
  JOIN {SILVER}.silver_work_orders w
    ON w.machine_id = g.machine_id AND w.completed_ts_clean IS NOT NULL
   AND to_date(w.completed_ts_clean) <= g.activity_date
  GROUP BY g.machine_id, g.activity_date
)
SELECT g.machine_id, g.activity_date, g.criticality, g.machine_type,
       coalesce(m.reading_count, 0)    AS reading_count,
       coalesce(m.abnormal_count, 0)   AS abnormal_count,
       m.abnormal_reading_rate,
       coalesce(m.failure_count, 0)    AS failures_today,
       coalesce(m.downtime_minutes, 0) AS downtime_minutes_today,
       s.avg_temperature_c, s.max_temperature_c, s.avg_vibration_mm_s, s.max_vibration_mm_s,
       s.avg_pressure_bar, s.max_pressure_bar,
       CAST(coalesce(s.any_drift_flag, false) AS INT)    AS any_drift_flag,
       datediff(g.activity_date, to_date(lm.last_maint_ts)) AS days_since_last_maintenance,
       CASE WHEN date_add(g.activity_date, 1) BETWEEN fw.first_day AND fw.last_day
            THEN CAST(f.machine_id IS NOT NULL AS INT) END AS {LABEL}
FROM grid g
CROSS JOIN failure_window fw
LEFT JOIN {GOLD}.mart_sensor_daily m ON m.machine_id = g.machine_id AND m.activity_date = g.activity_date
LEFT JOIN {GOLD}.vw_sensor_drift s   ON s.machine_id = g.machine_id AND s.activity_date = g.activity_date
LEFT JOIN last_maintenance lm        ON lm.machine_id = g.machine_id AND lm.activity_date = g.activity_date
LEFT JOIN failure_days f             ON f.machine_id = g.machine_id AND f.fail_date = date_add(g.activity_date, 1)
""")

display(spark.table(FEATURE_TABLE).agg(
    F.count("*").alias("rows"),
    F.countDistinct("machine_id").alias("machines"),
    F.count(LABEL).alias("labelled_rows"),
    F.sum(LABEL).alias("positives")))

## Step 2 – Look at the training data
Only labelled rows are used for training.

* Lists the input columns: number columns (readings, sensor values, etc.) and text columns (criticality, machine type).
* Reads only the rows that **have a label** from the feature table and turns them into a pandas table for training.
* Prints how many labelled rows and days there are, and what share of machines fail vs. do not fail.
* Removes columns that have the **same value in every row**, because they cannot help the model tell machines apart.
* Shows basic statistics (count, mean, min, max, ...) for each remaining number column.

In [0]:
NUMERIC = ["reading_count", "abnormal_count", "abnormal_reading_rate", "failures_today", "downtime_minutes_today",
           "avg_temperature_c", "max_temperature_c", "avg_vibration_mm_s", "max_vibration_mm_s",
           "avg_pressure_bar", "max_pressure_bar", "any_drift_flag", "days_since_last_maintenance"]
CATEGORICAL = ["criticality", "machine_type"]

train_pdf = spark.table(FEATURE_TABLE).filter(f"{LABEL} IS NOT NULL").toPandas()
train_pdf[NUMERIC] = train_pdf[NUMERIC].astype(float)
train_pdf[LABEL] = train_pdf[LABEL].astype(int)

print("labelled rows:", len(train_pdf), "| labelled days:", train_pdf["activity_date"].nunique())
print("class share (1 = fails next day):", train_pdf[LABEL].value_counts(normalize=True).round(3).to_dict())

# a feature with only one value carries no information, so it is left out
constant = [c for c in NUMERIC if train_pdf[c].nunique(dropna=True) <= 1]
NUMERIC_USED = [c for c in NUMERIC if c not in constant]
FEATURES = NUMERIC_USED + CATEGORICAL
print("left out (constant on the labelled day):", constant)
print("features used:", FEATURES)

display(train_pdf[NUMERIC_USED + [LABEL]].describe().T.reset_index())

## Step 3 – Train two models and log them to MLflow
* **Split:** a time-based split (train on earlier days, test on later days) is used when there are at least 2 labelled days. Here there is only **1 labelled day**, so 30% of the **machines** are held out instead (stratified, each machine appears once).
* **Class imbalance:** both models use `class_weight="balanced"`.
* **Preprocessing** (fill missing values, scale numbers, one-hot encode categories) is inside the sklearn `Pipeline`, so the logged model takes the raw feature columns.
* **Logged per run:** parameters, precision / recall / F1 / ROC-AUC, confusion matrix image and feature importance.

* Splits the data into a **training part** (used to learn) and a **test part** (used to check how good the model is). If there are several days it splits by date; with only one day it keeps 30% of the machines aside for testing.
* `build_pipeline` wraps each model with preparation steps: fill missing numbers with the average, put numbers on the same scale, and turn text categories into 0/1 columns.
* `feature_importance` reports which inputs matter most to the model.
* Trains **two models**: logistic regression and random forest. Both give extra weight to the smaller class so it is not ignored.
* For each model it calculates precision, recall, F1 and ROC-AUC on the test part, and saves to MLflow: the settings, the scores, a confusion-matrix picture and the feature-importance table.
* Shows a small table comparing the two models.

In [0]:
X, y = train_pdf[FEATURES], train_pdf[LABEL]

days = sorted(train_pdf["activity_date"].unique())
if len(days) >= 2:
    cutoff = days[int(len(days) * 0.7)]
    train_mask = train_pdf["activity_date"] < cutoff
    X_train, X_test, y_train, y_test = X[train_mask], X[~train_mask], y[train_mask], y[~train_mask]
    SPLIT = f"time_based (test from {cutoff})"
else:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
    SPLIT = "machine_holdout_30pct (only 1 labelled day)"
print("split:", SPLIT, "| train rows:", len(X_train), "| test rows:", len(X_test))


def build_pipeline(model):
    prep = ColumnTransformer([
        ("num", Pipeline([("fill", SimpleImputer(strategy="mean")), ("scale", StandardScaler())]), NUMERIC_USED),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])
    return Pipeline([("prep", prep), ("model", model)])


def feature_importance(pipe):
    names = pipe.named_steps["prep"].get_feature_names_out()
    model = pipe.named_steps["model"]
    values = model.feature_importances_ if hasattr(model, "feature_importances_") else np.abs(model.coef_[0])
    return pd.DataFrame({"feature": names, "importance": values}).sort_values("importance", ascending=False)


models = {
    "logistic_regression": LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=300, max_depth=4, min_samples_leaf=3,
                                            class_weight="balanced", random_state=42),
}

scores = []
for name, model in models.items():
    with mlflow.start_run(run_name=f"candidate_{name}"):
        pipe = build_pipeline(model).fit(X_train, y_train)
        prob = pipe.predict_proba(X_test)[:, 1]
        pred = (prob >= 0.5).astype(int)
        metrics = {"precision": precision_score(y_test, pred, zero_division=0),
                   "recall":    recall_score(y_test, pred, zero_division=0),
                   "f1":        f1_score(y_test, pred, zero_division=0),
                   "roc_auc":   roc_auc_score(y_test, prob)}

        mlflow.log_params({"model": name, "split": SPLIT, "n_features": len(FEATURES), "threshold": 0.5})
        mlflow.log_metrics(metrics)

        fig, ax = plt.subplots(figsize=(4, 4))
        ConfusionMatrixDisplay(confusion_matrix(y_test, pred, labels=[0, 1]),
                               display_labels=["no failure", "failure"]).plot(ax=ax, colorbar=False)
        ax.set_title(name)
        mlflow.log_figure(fig, "confusion_matrix.png")
        plt.close(fig)
        mlflow.log_table(feature_importance(pipe), "feature_importance.json")

        scores.append({"model": name, **{k: round(v, 4) for k, v in metrics.items()}})

scores_df = pd.DataFrame(scores)
display(scores_df)

## Step 4 – Log the best model, register it in Unity Catalog and set the alias `champion`
The best model (highest ROC-AUC, then F1) is trained again on **all** labelled rows and logged with a signature and an input example. Its holdout metrics from Step 3 are logged with it.
Every run of this cell creates a new model version, and the alias `champion` always points to the newest one.

* Picks the **best model** from Step 3 (highest ROC-AUC, and F1 if there is a tie).
* Trains a fresh copy of that model on **all** labelled rows.
* Records a **signature** (what input columns the model expects and what it returns) and a small input example.
* Starts a final MLflow run and saves the model, its test scores from Step 3 and some tags (project, stage, data).
* **Registers** the model in Unity Catalog as `gold.failure_risk_model` (each run creates a new version).
* Puts the label (alias) **`champion`** on this new version, so other code can always load "the current best model" by that name.

In [0]:
best = scores_df.sort_values(["roc_auc", "f1"], ascending=False).iloc[0]
BEST = best["model"]
print("best model:", BEST, best.to_dict())

final_pipe = build_pipeline(clone(models[BEST])).fit(X, y)     # fresh copy, trained on all labelled rows
signature = infer_signature(X, final_pipe.predict_proba(X))

with mlflow.start_run(run_name=f"final_{BEST}"):
    mlflow.log_params({"model": BEST, "split": SPLIT, "n_features": len(FEATURES)})
    mlflow.log_metrics({f"holdout_{k}": best[k] for k in ["precision", "recall", "f1", "roc_auc"]})
    mlflow.set_tags({"project": "capstone05", "stage": "pilot", "data": "synthetic"})
    model_info = mlflow.sklearn.log_model(final_pipe, name="model", signature=signature,
                                          input_example=X.head(3), pyfunc_predict_fn="predict_proba")

registered = mlflow.register_model(model_info.model_uri, MODEL_NAME)
client = MlflowClient(registry_uri="databricks-uc")
client.set_registered_model_alias(MODEL_NAME, "champion", registered.version)
print(f"registered {MODEL_NAME} version {registered.version} with alias 'champion'")

## Step 5 – Score all machines and create the dashboard view
* The model is loaded **from the registry by alias**: `models:/capstone_5iot.gold.failure_risk_model@champion`.
* Scores are for the latest day in the feature table, for all 150 machines.
* Risk bands: **HIGH** ≥ 0.6, **MEDIUM** ≥ 0.3, **LOW** < 0.3.
* The MERGE on `(machine_id, score_date)` makes the cell safe to re-run.

* Loads the **champion** model from Unity Catalog.
* Takes the rows for the **latest day** in the feature table (all machines).
* Asks the model for each machine's **chance of failing tomorrow** (`risk_score`, from 0 to 1).
* Turns the score into a simple band: **HIGH** (0.6 or more), **MEDIUM** (0.3 to 0.6) or **LOW** (below 0.3).
* Creates the table `gold.machine_failure_risk` if it does not exist, then **MERGEs** the new scores into it: existing rows for the same machine and date are updated, new ones are added. So running it again does not create duplicates.
* Shows how many machines fall into each band and their average score.

In [0]:
champion = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")

latest_day = spark.table(FEATURE_TABLE).agg(F.max("activity_date")).first()[0]
score_pdf = spark.table(FEATURE_TABLE).filter(F.col("activity_date") == F.lit(latest_day)).toPandas()
score_pdf[NUMERIC] = score_pdf[NUMERIC].astype(float)

score_pdf["risk_score"] = champion.predict_proba(score_pdf[FEATURES])[:, 1].round(4)
score_pdf["risk_band"] = np.select([score_pdf["risk_score"] >= 0.6, score_pdf["risk_score"] >= 0.3], ["HIGH", "MEDIUM"], "LOW")
score_pdf = score_pdf.rename(columns={"activity_date": "score_date"})[["machine_id", "score_date", "risk_score", "risk_band"]]

spark.sql(f"""CREATE TABLE IF NOT EXISTS {RISK_TABLE}
              (machine_id STRING, score_date DATE, risk_score DOUBLE, risk_band STRING)
              COMMENT 'Day 5 pilot next-day failure risk per machine (see notebook 7_4 limitations)'""")
spark.createDataFrame(score_pdf).createOrReplaceTempView("new_scores")
spark.sql(f"""MERGE INTO {RISK_TABLE} t USING new_scores s
              ON t.machine_id = s.machine_id AND t.score_date = s.score_date
              WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")

print("score date:", latest_day)
display(spark.table(RISK_TABLE).filter(F.col("score_date") == F.lit(latest_day))
             .groupBy("risk_band").agg(F.count("*").alias("machines"), F.round(F.avg("risk_score"), 4).alias("avg_score"))
             .orderBy("risk_band"))

* Creates the view `gold.vw_machine_failure_risk` for the dashboard.
* It joins each machine's risk score with useful machine details: plant, region, machine type, criticality, status, and its failures, downtime and availability for the period.
* It only shows the **most recent score date**, so the dashboard always displays the latest risk.

In [0]:
%sql
CREATE OR REPLACE VIEW capstone_5iot.gold.vw_machine_failure_risk
COMMENT 'Dashboard: pilot next-day failure risk per machine with machine and plant attributes (latest score date)'
AS
SELECT r.machine_id, r.score_date, r.risk_score, r.risk_band,
       m.plant_id, m.plant_name, m.region, m.machine_type, m.criticality, m.operating_status,
       m.failures         AS failures_in_period,
       m.downtime_minutes AS downtime_minutes_in_period,
       m.availability
FROM capstone_5iot.gold.machine_failure_risk r
JOIN capstone_5iot.gold.vw_machine_reliability m ON m.machine_id = r.machine_id
WHERE r.score_date = (SELECT max(score_date) FROM capstone_5iot.gold.machine_failure_risk);

* Shows the **10 machines with the highest risk score** from the dashboard view, as a quick check that everything worked.

In [0]:
%sql
SELECT * FROM capstone_5iot.gold.vw_machine_failure_risk ORDER BY risk_score DESC LIMIT 10;

## Step 6 – Limitations (say this in the demo)
* **Tiny label window.** The failure log covers only 2 of the 57 days. With a next-day label that leaves **1 labelled day = 150 rows** for training, and the test set is about 45 machines.
* **No out-of-time test.** With one labelled day the holdout is by machine. It tests other machines on the same day, not future days.
* **Near-chance accuracy.** ROC-AUC is around 0.55, close to random. Precision and recall look moderate only because about 62% of the labelled machines did fail the next day.
* **Weak features.** Drift flags and days-since-maintenance are constant on the labelled day, so they are left out.
* **Scores are a ranking, not a probability.** They are inflated by the 62% base rate. Use the band to compare machines, not as an alarm.
* **Not production-ready.** It demonstrates the MLflow pipeline. Real use needs a full-period failure log, an out-of-time backtest, calibration and threshold tuning against maintenance cost.

**Where to show MLflow in the demo:** *Experiments → capstone05_failure_risk* (compare the two candidate runs, open the confusion matrix) and *Catalog → capstone_5iot → gold → Models → failure_risk_model* (versions and the `champion` alias).